In [ ]:
!pip install -q groq chromadb sentence_transformers

import chromadb
from groq import Groq
from google.colab import userdata
from sentence_transformers import SentenceTransformer

# 1. Setup Clients
groq_key = userdata.get('GROQ_API_KEY')
client = Groq(api_key=groq_key)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
db_client = chromadb.Client()

# 2. Setup your DB (Day 5 memory)
collection = db_client.get_or_create_collection(name="college_subjects")
subjects = [
    "Data Structures: Focus on Trees and Graphs. Big O notation is vital.",
    "Operating Systems: Focus on Memory Management and CPU Scheduling."
]
collection.add(
    documents=subjects,
    ids=["id1", "id2"],
    embeddings=embed_model.encode(subjects).tolist()
)

# 3. THE RAG FUNCTION (Groq Powered)
def rag_generate_groq(query):
    # STEP A: Retrieve context
    query_vec = embed_model.encode(query).tolist()
    results = collection.query(query_embeddings=[query_vec], n_results=1)
    context = results['documents'][0][0]

    # STEP B: Augment & Generate
    # We use llama-3.3-70b-versatile for high intelligence
    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": f"You are a study assistant. Use ONLY this context: {context}"
            },
            {
                "role": "user",
                "content": query,
            }
        ],
        model="llama-3.3-70b-versatile",
    )

    return chat_completion.choices[0].message.content

# 4. TEST IT
query_text = "How should I study for OS?"
print(f"🔍 Searching and Generating with Groq...")
print(f"🤖 Groq Response: \n{rag_generate_groq(query_text)}")